# Baldr, NumPyro, and Distrax

This notebook compares the three JAX distribution implementations under the
same inputs, dtype, synchronization, and timing procedure.

The comparison is narrower than a comparison of the packages as a whole.
NumPyro is a probabilistic-programming system, Distrax is a general JAX
distribution library, and Baldr is optimized for small inference hot paths.
Here we compare only their common numerical kernels.

The notebook reports method availability, eager and warm-JIT timings,
one-time compilation cost, array scaling, numerical agreement with SciPy, a
fused log-density, and a fused prior transform where inverse CDFs are available.

Set PROFILE to "full" for longer, more stable timings. Absolute results are
specific to the recorded machine and package versions.

In [ ]:
import importlib.metadata
import platform
import statistics
import sys
import time

import distrax
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import numpyro.distributions as numpyro_dist
import pandas as pd
from scipy import stats

from baldr import Beta, Gamma, Normal

jax.config.update("jax_enable_x64", True)

PROFILE = "quick"  # Change to "full" for more stable timings.
REPEAT = 5 if PROFILE == "quick" else 15
NUMBER_ARRAY = 30 if PROFILE == "quick" else 200
TARGET_ELEMENTS = 100_000 if PROFILE == "quick" else 2_000_000
ARRAY_SIZES = (8, 256, 4_096) if PROFILE == "quick" else (
    8,
    32,
    256,
    4_096,
    65_536,
    1_000_000,
)

In [ ]:
def package_version(name):
    """Return an installed package version.

    Parameters
    ----------
    name : str
        Distribution name understood by package metadata.

    Returns
    -------
    str
        Installed version or "not installed".
    """

    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return "not installed"


metadata = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "processor": platform.processor() or "not reported",
    **{
        name: package_version(name)
        for name in (
            "baldr",
            "jax",
            "jaxlib",
            "numpyro",
            "distrax",
            "numpy",
            "scipy",
        )
    },
}
pd.Series(metadata, name="version")

## Equivalent distributions and method adapters

The parameterizations are matched explicitly. Normal uses the same loc and
scale. Beta uses the standard support [0, 1]. Gamma uses loc=0, and Baldr's
scale is converted to the rate=1/scale convention used by NumPyro and Distrax.
NumPyro argument validation is disabled because Baldr validates once during
construction and the benchmark measures repeated evaluation.

The adapters translate naming only: logpdf to log_prob and ppf to icdf or
quantile. NumPyro has no separate pdf method, so its PDF is exp(log_prob).

In [ ]:
FACTORIES = {
    "Normal": {
        "baldr": lambda: Normal(loc=0.3, scale=1.7, backend="jax"),
        "numpyro": lambda: numpyro_dist.Normal(
            loc=0.3,
            scale=1.7,
            validate_args=False,
        ),
        "distrax": lambda: distrax.Normal(loc=0.3, scale=1.7),
    },
    "Beta": {
        "baldr": lambda: Beta(
            a=2.3,
            b=5.1,
            loc=0.0,
            scale=1.0,
            backend="jax",
        ),
        "numpyro": lambda: numpyro_dist.Beta(
            concentration1=2.3,
            concentration0=5.1,
            validate_args=False,
        ),
        "distrax": lambda: distrax.Beta(alpha=2.3, beta=5.1),
    },
    "Gamma": {
        "baldr": lambda: Gamma(
            a=2.7,
            loc=0.0,
            scale=1.3,
            backend="jax",
        ),
        "numpyro": lambda: numpyro_dist.Gamma(
            concentration=2.7,
            rate=1.0 / 1.3,
            validate_args=False,
        ),
        "distrax": lambda: distrax.Gamma(
            concentration=2.7,
            rate=1.0 / 1.3,
        ),
    },
}

METHODS = ("logpdf", "pdf", "cdf", "ppf")


def method_adapters(library, distribution):
    """Return common method names for one distribution object.

    Parameters
    ----------
    library : {"baldr", "numpyro", "distrax"}
        Source library.
    distribution : object
        Constructed distribution instance.

    Returns
    -------
    dict
        Mapping from Baldr-style names to callables. Unsupported methods are
        absent.
    """

    if library == "baldr":
        return {
            "logpdf": distribution.logpdf,
            "pdf": distribution.pdf,
            "cdf": distribution.cdf,
            "ppf": distribution.ppf,
        }

    if library == "numpyro":
        methods = {
            "logpdf": distribution.log_prob,
            "pdf": lambda value: jnp.exp(distribution.log_prob(value)),
            "cdf": distribution.cdf,
        }
        if hasattr(distribution, "icdf"):
            methods["ppf"] = distribution.icdf
        return methods

    methods = {
        "logpdf": distribution.log_prob,
        "pdf": distribution.prob,
        "cdf": distribution.cdf,
    }
    if hasattr(distribution, "quantile"):
        methods["ppf"] = distribution.quantile
    return methods


capability_rows = []
for distribution_name, factories in FACTORIES.items():
    for library, factory in factories.items():
        adapters = method_adapters(library, factory())
        for method_name in METHODS:
            capability_rows.append(
                {
                    "distribution": distribution_name,
                    "library": library,
                    "method": method_name,
                    "available": method_name in adapters,
                }
            )

capabilities = pd.DataFrame(capability_rows)
capabilities.pivot_table(
    index=["distribution", "method"],
    columns="library",
    values="available",
    aggfunc="first",
)

## Timing helpers

JAX dispatch is asynchronous, so every result is synchronized. Eager functions
are warmed once to exclude one-time JAX initialization. For JIT, the first
synchronized call is reported separately as an approximate trace-and-compile
cost; later calls measure warm execution. Construction is excluded because
inference normally constructs a distribution once and evaluates it many times.

In [ ]:
def synchronize(value):
    """Wait for a possibly asynchronous JAX result.

    Parameters
    ----------
    value : object
        Function result.

    Returns
    -------
    object
        Synchronized result.
    """

    blocker = getattr(value, "block_until_ready", None)
    return blocker() if blocker is not None else value


def time_call(function, argument, *, repeat, number):
    """Time synchronized repeated calls.

    Parameters
    ----------
    function : callable
        Single-argument function to benchmark.
    argument : array-like
        Function input.
    repeat : int
        Number of timing samples.
    number : int
        Calls per timing sample.

    Returns
    -------
    dict
        Median, minimum, and maximum nanoseconds per call.
    """

    samples = []
    for _ in range(repeat):
        start = time.perf_counter_ns()
        for _ in range(number):
            synchronize(function(argument))
        samples.append((time.perf_counter_ns() - start) / number)
    return {
        "median_ns": statistics.median(samples),
        "minimum_ns": min(samples),
        "maximum_ns": max(samples),
    }


def benchmark_argument(distribution_name, method_name, size):
    """Create a valid benchmark input.

    Parameters
    ----------
    distribution_name : str
        Distribution family.
    method_name : str
        Common method name.
    size : int
        Number of input elements.

    Returns
    -------
    jax.Array
        Float64 input array.
    """

    if method_name == "ppf":
        return jnp.linspace(0.01, 0.99, size, dtype=jnp.float64)
    if distribution_name == "Normal":
        return jnp.linspace(-3.0, 3.0, size, dtype=jnp.float64)
    if distribution_name == "Beta":
        return jnp.linspace(0.01, 0.99, size, dtype=jnp.float64)
    return jnp.linspace(0.01, 10.0, size, dtype=jnp.float64)

## Eager and JIT method benchmarks

Eager means calling the distribution method directly from Python without
wrapping the whole method in jax.jit. Warm JIT means that the compiled function
has already run once. This distinction matters because a fast compiled kernel
can still be a poor choice for isolated calls if compilation or dispatch
dominates.

In [ ]:
timing_rows = []
for distribution_name, factories in FACTORIES.items():
    for library, factory in factories.items():
        distribution = factory()
        adapters = method_adapters(library, distribution)
        for method_name, function in adapters.items():
            for size in ARRAY_SIZES:
                argument = benchmark_argument(
                    distribution_name,
                    method_name,
                    size,
                )
                number = max(
                    1,
                    min(NUMBER_ARRAY, TARGET_ELEMENTS // size),
                )

                synchronize(function(argument))
                eager_timing = time_call(
                    function,
                    argument,
                    repeat=REPEAT,
                    number=number,
                )
                timing_rows.append(
                    {
                        "distribution": distribution_name,
                        "method": method_name,
                        "library": library,
                        "mode": "eager",
                        "size": size,
                        "compilation_ns": 0,
                        **eager_timing,
                    }
                )

                compiled = jax.jit(function)
                start = time.perf_counter_ns()
                synchronize(compiled(argument))
                compilation_ns = time.perf_counter_ns() - start
                jit_timing = time_call(
                    compiled,
                    argument,
                    repeat=REPEAT,
                    number=number,
                )
                timing_rows.append(
                    {
                        "distribution": distribution_name,
                        "method": method_name,
                        "library": library,
                        "mode": "warm_jit",
                        "size": size,
                        "compilation_ns": compilation_ns,
                        **jit_timing,
                    }
                )

timings = pd.DataFrame(timing_rows)
timings["median_ns_per_element"] = timings["median_ns"] / timings["size"]
timings.head()

In [ ]:
normal_logpdf = timings.query(
    "distribution == 'Normal' and method == 'logpdf'"
)
figure, axes = plt.subplots(1, 2, figsize=(12, 4))
for mode, axis in zip(("eager", "warm_jit"), axes):
    subset = normal_logpdf.query("mode == @mode")
    for library, group in subset.groupby("library"):
        axis.loglog(
            group["size"],
            group["median_ns_per_element"],
            marker="o",
            label=library,
        )
    axis.set_title(mode.replace("_", " ").title())
    axis.set_xlabel("Array size")
    axis.set_ylabel("Median time [ns / element]")
    axis.legend()
figure.suptitle("Normal log-density performance")
figure.tight_layout()

In [ ]:
largest_size = max(ARRAY_SIZES)
timings.query(
    "mode == 'warm_jit' and size == @largest_size"
).pivot_table(
    index=["distribution", "method"],
    columns="library",
    values="median_ns_per_element",
).round(3)

In [ ]:
compilation = timings.query("mode == 'warm_jit'").pivot_table(
    index=["distribution", "method", "size"],
    columns="library",
    values="compilation_ns",
)
(compilation / 1e6).round(2)

## Numerical agreement with SciPy

Timing only equivalent calculations is essential. The following comparison
uses float64 and SciPy as a reference. It reports finite-value absolute and
relative errors separately from NaN/infinity agreement. PPF tails are included
where the library provides an inverse CDF.

In [ ]:
SCIPY_DISTRIBUTIONS = {
    "Normal": stats.norm(loc=0.3, scale=1.7),
    "Beta": stats.beta(a=2.3, b=5.1),
    "Gamma": stats.gamma(a=2.7, scale=1.3),
}

ACCURACY_INPUTS = {
    "Normal": np.linspace(-8.0, 8.0, 1_001),
    "Beta": np.linspace(1e-12, 1.0 - 1e-12, 1_001),
    "Gamma": np.geomspace(1e-12, 50.0, 1_001),
}

TAIL_PROBABILITIES = np.array(
    [
        1e-12,
        1e-9,
        1e-6,
        1e-3,
        0.1,
        0.5,
        0.9,
        1.0 - 1e-3,
        1.0 - 1e-6,
        1.0 - 1e-9,
        1.0 - 1e-12,
    ]
)


def finite_errors(actual, reference):
    """Summarize finite and special-value agreement.

    Parameters
    ----------
    actual : array-like
        Values from the tested implementation.
    reference : array-like
        Reference values.

    Returns
    -------
    tuple
        Maximum absolute error, maximum relative error, and whether all
        non-finite values agree.
    """

    actual = np.asarray(actual, dtype=float)
    reference = np.asarray(reference, dtype=float)
    finite = np.isfinite(actual) & np.isfinite(reference)
    absolute = (
        np.max(np.abs(actual[finite] - reference[finite]))
        if np.any(finite)
        else 0.0
    )
    denominator = np.maximum(np.abs(reference[finite]), 1e-300)
    relative = (
        np.max(
            np.abs(actual[finite] - reference[finite]) / denominator
        )
        if np.any(finite)
        else 0.0
    )
    special_match = np.all(
        (actual[~finite] == reference[~finite])
        | (np.isnan(actual[~finite]) & np.isnan(reference[~finite]))
    )
    return absolute, relative, special_match

In [ ]:
accuracy_rows = []
for distribution_name, factories in FACTORIES.items():
    scipy_distribution = SCIPY_DISTRIBUTIONS[distribution_name]
    for library, factory in factories.items():
        adapters = method_adapters(library, factory())
        for method_name in METHODS:
            if method_name not in adapters:
                accuracy_rows.append(
                    {
                        "distribution": distribution_name,
                        "method": method_name,
                        "library": library,
                        "supported": False,
                        "max_absolute_error": np.nan,
                        "max_relative_error": np.nan,
                        "special_values_match": False,
                    }
                )
                continue

            values = (
                TAIL_PROBABILITIES
                if method_name == "ppf"
                else ACCURACY_INPUTS[distribution_name]
            )
            reference = getattr(
                scipy_distribution,
                method_name,
            )(values)
            actual = synchronize(
                jax.jit(adapters[method_name])(
                    jnp.asarray(values, dtype=jnp.float64)
                )
            )
            absolute, relative, special_match = finite_errors(
                actual,
                reference,
            )
            accuracy_rows.append(
                {
                    "distribution": distribution_name,
                    "method": method_name,
                    "library": library,
                    "supported": True,
                    "max_absolute_error": absolute,
                    "max_relative_error": relative,
                    "special_values_match": special_match,
                }
            )

accuracy = pd.DataFrame(accuracy_rows)
accuracy

In [ ]:
accuracy.query("supported").pivot_table(
    index=["distribution", "method"],
    columns="library",
    values="max_absolute_error",
).style.format("{:.3e}")

## Fused inference workloads

Method-level timings are diagnostic, but Baldr is intended for use inside a
larger compiled calculation. Two fused examples are therefore more relevant:

1. A Normal + Beta + Gamma log-density, available in all three libraries.
2. A three-parameter prior transform, available only where all three inverse
   CDFs exist.

Each complete calculation is JIT-compiled as one unit. This is analogous to
timing an assembled engine rather than timing each piston across a Python call
boundary.

In [ ]:
def make_fused_logdensity(library):
    """Construct one fused three-distribution log-density.

    Parameters
    ----------
    library : {"baldr", "numpyro", "distrax"}
        Source library.

    Returns
    -------
    callable
        Function accepting a three-element vector.
    """

    methods = [
        method_adapters(library, FACTORIES[name][library]())["logpdf"]
        for name in ("Normal", "Beta", "Gamma")
    ]

    def fused(values):
        """Evaluate the sum of three log densities.

        Parameters
        ----------
        values : array-like
            Normal, Beta, and Gamma evaluation points.

        Returns
        -------
        jax.Array
            Sum of the three log densities.
        """

        return sum(function(value) for function, value in zip(methods, values))

    return fused


def make_fused_transform(library):
    """Construct one fused three-distribution prior transform.

    Parameters
    ----------
    library : {"baldr", "numpyro", "distrax"}
        Source library.

    Returns
    -------
    callable or None
        Transform accepting three unit-cube values, or None when at least one
        inverse CDF is unavailable.
    """

    adapters = [
        method_adapters(library, FACTORIES[name][library]())
        for name in ("Normal", "Beta", "Gamma")
    ]
    if not all("ppf" in methods for methods in adapters):
        return None
    quantiles = [methods["ppf"] for methods in adapters]

    def fused(probabilities):
        """Transform three unit-cube values.

        Parameters
        ----------
        probabilities : array-like
            Three probabilities in the unit interval.

        Returns
        -------
        jax.Array
            Normal, Beta, and Gamma quantiles.
        """

        return jnp.stack(
            [
                quantile(probability)
                for quantile, probability in zip(
                    quantiles,
                    probabilities,
                )
            ]
        )

    return fused


def benchmark_fused(function, argument):
    """Measure compilation and warm execution of a fused function.

    Parameters
    ----------
    function : callable
        Fused JAX function.
    argument : array-like
        Function input.

    Returns
    -------
    dict
        Compilation milliseconds and warm nanoseconds per call.
    """

    compiled = jax.jit(function)
    start = time.perf_counter_ns()
    synchronize(compiled(argument))
    compilation_ns = time.perf_counter_ns() - start
    timing = time_call(
        compiled,
        argument,
        repeat=REPEAT,
        number=1_000 if PROFILE == "quick" else 10_000,
    )
    return {
        "compilation_ms": compilation_ns / 1e6,
        "warm_median_ns": timing["median_ns"],
    }

In [ ]:
fused_rows = []
logdensity_argument = jnp.array([0.8, 0.35, 2.1], dtype=jnp.float64)
probability_argument = jnp.array([0.2, 0.5, 0.8], dtype=jnp.float64)

for library in ("baldr", "numpyro", "distrax"):
    fused_rows.append(
        {
            "workload": "logdensity",
            "library": library,
            "available": True,
            **benchmark_fused(
                make_fused_logdensity(library),
                logdensity_argument,
            ),
        }
    )

    transform = make_fused_transform(library)
    if transform is None:
        fused_rows.append(
            {
                "workload": "prior_transform",
                "library": library,
                "available": False,
                "compilation_ms": np.nan,
                "warm_median_ns": np.nan,
            }
        )
    else:
        fused_rows.append(
            {
                "workload": "prior_transform",
                "library": library,
                "available": True,
                **benchmark_fused(transform, probability_argument),
            }
        )

fused_results = pd.DataFrame(fused_rows)
fused_results

## Interpretation

- Compare warm JIT timings when the intended calculation will be compiled.
  Eager timings mainly quantify Python and framework dispatch overhead.
- Treat compilation time separately. A kernel that wins after compilation may
  lose when it is compiled repeatedly.
- For Dynesty-like scalar callbacks, also compare Baldr's scalar backend in the
  main performance notebook; the three libraries here all use JAX arrays.
- NumPyro's PDF is derived from exp(log_prob), so that row measures the natural
  numerical route rather than a dedicated method.
- The Distrax version recorded by this run lacks inverse CDFs for the Normal, Beta, and Gamma classes
  used here. It can participate in likelihood and log-density comparisons but
  not this PPF-based prior-transform comparison.
- Numerical agreement must be considered alongside speed, especially for Beta
  and Gamma tail quantiles.
- These results compare distribution kernels, not NumPyro's inference machinery
  or Distrax's broader bijector and distribution APIs.